In [10]:
import os
import re
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from scipy.stats import spearmanr
import plotly.io as pio
from typing import Dict, List, Optional

pio.renderers.default = "plotly_mimetype"

# --- Helper Functions ---
def parse_config(file_path: str) -> Dict[str, str]:
    """Parses a 'key: value' or 'key = value' configuration file."""
    params = {}
    try:
        with open(file_path, 'r') as f:
            for line in f:
                line = line.strip()
                if not line or line.startswith('#'):
                    continue
                parts = re.split(r'[:=]', line, 1)
                if len(parts) == 2:
                    key, value = parts
                    params[key.strip().lower()] = value.strip()
    except Exception:
        pass
    return params

def parse_runtime(time_str: str) -> float:
    """Parses 'H:MM:SS.ffffff' into seconds."""
    if not time_str: return 0.0
    try:
        parts = time_str.split(':')
        return int(parts[0]) * 3600 + int(parts[1]) * 60 + float(parts[2])
    except (ValueError, IndexError):
        return 0.0

def parse_workload_params(workload_name: str) -> dict:
    """Parse model/strategy params from workload directory name.
    Format: d{dmodel}_L{layers}_seq{seq}_b{batch}_mb{mb}_{dp}_{tp}_1_{pp}_{ws}
    """
    m = re.match(
        r'd(\d+)_L(\d+)_seq(\d+)_b(\d+)_mb(\d+)_(\d+)_(\d+)_(\d+)_(\d+)_(\d+)',
        workload_name
    )
    if not m:
        return {}
    return {
        'd_model': int(m.group(1)),
        'num_stacks': int(m.group(2)),
        'seq_len': int(m.group(3)),
        'batch': int(m.group(4)),
        'micro_batch': int(m.group(5)),
        'dp': int(m.group(6)),
        'tp': int(m.group(7)),
        'sp': int(m.group(8)),
        'pp': int(m.group(9)),
        'weight_sharded': int(m.group(10)),
    }

# --- Main Data Collection ---

BASE_OUTPUT_DIR = '/app/astra-sim/upc/output/comparison_run/'
EXPERIMENT = 'experiment8'
TOPOLOGY = 'FoldedClosECMP128'

topo_base = os.path.join(BASE_OUTPUT_DIR, EXPERIMENT, TOPOLOGY)
all_results = []

if not os.path.isdir(topo_base):
    print(f"Output directory not found: {topo_base}")
else:
    for npu_dir in sorted(os.listdir(topo_base)):
        npu_path = os.path.join(topo_base, npu_dir)
        if not os.path.isdir(npu_path) or not npu_dir.startswith('npu_'):
            continue
        npu_count = int(npu_dir.split('_')[1])

        for workload_name in os.listdir(npu_path):
            workload_path = os.path.join(npu_path, workload_name)
            if not os.path.isdir(workload_path):
                continue

            wparams = parse_workload_params(workload_name)
            workload_results = {
                'workload': workload_name,
                'npu_count': npu_count,
                **wparams,
            }

            g2_exec_times, g2_sim_times = [], []
            ns3_exec_times, ns3_sim_times = [], []

            run_dirs = [d for d in os.listdir(workload_path) if d.startswith('run_')]
            for run_dir_name in run_dirs:
                run_path = os.path.join(workload_path, run_dir_name)

                sim_type = None
                for st in ['g2', 'ns3', 'analytical_unaware']:
                    if os.path.isdir(os.path.join(run_path, st)):
                        sim_type = st
                        break
                if not sim_type:
                    continue

                sim_path = os.path.join(run_path, sim_type)
                sim_name = {'g2': 'G2', 'ns3': 'NS3', 'analytical_unaware': 'Analytical'}[sim_type]

                # Extract estimated execution time
                timing_file = next(
                    (os.path.join(sim_path, f) for f in os.listdir(sim_path)
                     if 'trace_matched_timing.csv' in f), None
                )
                max_time_ns = None
                if timing_file:
                    try:
                        df_timing = pd.read_csv(timing_file)
                        if 'callback_tick' in df_timing.columns:
                            max_time_ns = df_timing['callback_tick'].max()
                    except Exception as e:
                        print(f"Error reading {timing_file}: {e}")
                else:
                    continue

                # Extract simulation time
                summary_params = parse_config(os.path.join(run_path, 'run_summary.txt'))
                sim_time_sec = parse_runtime(summary_params.get('total runtime', '0'))

                if sim_type == 'g2':
                    if max_time_ns is not None: g2_exec_times.append(max_time_ns)
                    if sim_time_sec: g2_sim_times.append(sim_time_sec)
                elif sim_type == 'ns3':
                    if max_time_ns is not None: ns3_exec_times.append(max_time_ns)
                    if sim_time_sec: ns3_sim_times.append(sim_time_sec)
                else:
                    workload_results[f'{sim_name}_Est_Exec_Time_ns'] = max_time_ns
                    workload_results[f'{sim_name}_Sim_Time_sec'] = sim_time_sec

            # Aggregate G2 stats
            if g2_exec_times:
                workload_results['G2_Est_Exec_Time_ns'] = sum(g2_exec_times) / len(g2_exec_times)
                workload_results['G2_Est_Exec_Time_ns_min'] = min(g2_exec_times)
                workload_results['G2_Est_Exec_Time_ns_max'] = max(g2_exec_times)
            if g2_sim_times:
                workload_results['G2_Sim_Time_sec'] = sum(g2_sim_times) / len(g2_sim_times)
                workload_results['G2_Sim_Time_sec_min'] = min(g2_sim_times)
                workload_results['G2_Sim_Time_sec_max'] = max(g2_sim_times)

            # Aggregate NS3 stats
            if ns3_exec_times:
                workload_results['NS3_Est_Exec_Time_ns'] = sum(ns3_exec_times) / len(ns3_exec_times)
                workload_results['NS3_Est_Exec_Time_ns_min'] = min(ns3_exec_times)
                workload_results['NS3_Est_Exec_Time_ns_max'] = max(ns3_exec_times)
            if ns3_sim_times:
                workload_results['NS3_Sim_Time_sec'] = sum(ns3_sim_times) / len(ns3_sim_times)
                workload_results['NS3_Sim_Time_sec_min'] = min(ns3_sim_times)
                workload_results['NS3_Sim_Time_sec_max'] = max(ns3_sim_times)

            if len(workload_results) > 3:
                all_results.append(workload_results)

print(f"Collected {len(all_results)} workload results")

Collected 256 workload results


In [11]:
# --- Build DataFrame and compute metrics ---

if not all_results:
    raise SystemExit("No results found. Run the simulations first.")

df = pd.DataFrame(all_results)

# Require all three simulators
required_cols = [
    'G2_Est_Exec_Time_ns', 'NS3_Est_Exec_Time_ns', 'Analytical_Est_Exec_Time_ns',
    'G2_Sim_Time_sec', 'NS3_Sim_Time_sec', 'Analytical_Sim_Time_sec',
]
df.dropna(subset=required_cols, inplace=True)
df.sort_values(by=['npu_count', 'workload'], inplace=True)

# Compute error & speedup
df['G2 Error (%)'] = ((df['G2_Est_Exec_Time_ns'] - df['NS3_Est_Exec_Time_ns']) / df['NS3_Est_Exec_Time_ns']) * 100
df['AU Error (%)'] = ((df['Analytical_Est_Exec_Time_ns'] - df['NS3_Est_Exec_Time_ns']) / df['NS3_Est_Exec_Time_ns']) * 100
df['G2 Speedup (x)'] = df['NS3_Sim_Time_sec'] / df['G2_Sim_Time_sec']
df['AU Speedup (x)'] = df['NS3_Sim_Time_sec'] / df['Analytical_Sim_Time_sec']

# Short workload label
df['short_workload'] = (
    df['workload']
    .str.replace('_', '-', regex=False)
)

print(f"Valid rows with all 3 simulators: {len(df)}")
print(f"NPU counts present: {sorted(df['npu_count'].unique())}")
print(f"d_model values: {sorted(df['d_model'].unique())}")
print(f"num_stacks values: {sorted(df['num_stacks'].unique())}")
print(f"seq_len values: {sorted(df['seq_len'].unique())}")

Valid rows with all 3 simulators: 227
NPU counts present: [2, 4, 8, 16, 32, 64]
d_model values: [512, 1024, 2048, 4096]
num_stacks values: [2, 4, 8, 16, 32]
seq_len values: [512, 1024, 2048, 4096]


In [12]:
# --- Plot 1: Per-NPU-count absolute execution time comparison ---

for npu in sorted(df['npu_count'].unique()):
    ndf = df[df['npu_count'] == npu].sort_values('NS3_Est_Exec_Time_ns').reset_index(drop=True)
    if ndf.empty:
        continue

    fig = go.Figure()

    # NS3 bars with error bars
    ns3_err_minus = ndf['NS3_Est_Exec_Time_ns'] - ndf.get('NS3_Est_Exec_Time_ns_min', ndf['NS3_Est_Exec_Time_ns'])
    ns3_err_plus = ndf.get('NS3_Est_Exec_Time_ns_max', ndf['NS3_Est_Exec_Time_ns']) - ndf['NS3_Est_Exec_Time_ns']
    fig.add_trace(go.Bar(
        x=ndf['short_workload'], y=ndf['NS3_Est_Exec_Time_ns'],
        name='NS3', marker_color='lightgray',
        error_y=dict(type='data', symmetric=False, array=ns3_err_plus, arrayminus=ns3_err_minus,
                     color='gray', thickness=1.5, width=4)
    ))

    # G2 bars with error bars
    g2_err_minus = ndf['G2_Est_Exec_Time_ns'] - ndf.get('G2_Est_Exec_Time_ns_min', ndf['G2_Est_Exec_Time_ns'])
    g2_err_plus = ndf.get('G2_Est_Exec_Time_ns_max', ndf['G2_Est_Exec_Time_ns']) - ndf['G2_Est_Exec_Time_ns']
    fig.add_trace(go.Bar(
        x=ndf['short_workload'], y=ndf['G2_Est_Exec_Time_ns'],
        name='G2', marker_color='skyblue',
        error_y=dict(type='data', symmetric=False, array=g2_err_plus, arrayminus=g2_err_minus,
                     color='blue', thickness=1.5, width=4)
    ))

    fig.add_trace(go.Bar(
        x=ndf['short_workload'], y=ndf['Analytical_Est_Exec_Time_ns'],
        name='Analytical', marker_color='salmon'
    ))

    fig.update_layout(
        title=f'Estimated Execution Time - {npu} NPUs',
        xaxis_title='Workload', yaxis_title='Estimated Execution Time (ns)',
        barmode='group', xaxis_tickangle=-45,
        template='plotly_white', font=dict(size=14),
        height=500, width=max(800, len(ndf) * 120),
    )
    fig.show()

In [13]:
# --- Plot 2: Normalized execution time (relative to NS3) per NPU count ---

for npu in sorted(df['npu_count'].unique()):
    ndf = df[df['npu_count'] == npu].sort_values('NS3_Est_Exec_Time_ns').reset_index(drop=True)
    if ndf.empty:
        continue

    g2_norm = (ndf['G2_Est_Exec_Time_ns'] / ndf['NS3_Est_Exec_Time_ns']) * 100
    au_norm = (ndf['Analytical_Est_Exec_Time_ns'] / ndf['NS3_Est_Exec_Time_ns']) * 100

    fig = go.Figure()
    fig.add_trace(go.Bar(x=ndf['short_workload'], y=[100]*len(ndf), name='NS3', marker_color='lightgray'))
    fig.add_trace(go.Bar(x=ndf['short_workload'], y=g2_norm, name='G2', marker_color='skyblue'))
    fig.add_trace(go.Bar(x=ndf['short_workload'], y=au_norm, name='Analytical', marker_color='salmon'))

    fig.update_layout(
        title=f'Normalized Execution Time (vs NS3) - {npu} NPUs',
        xaxis_title='Workload', yaxis_title='Relative Execution Time (%)',
        barmode='group', xaxis_tickangle=-45,
        template='plotly_white', font=dict(size=14),
        height=500, width=max(800, len(ndf) * 120),
    )
    fig.show()

In [14]:
# --- Plot 3: Error vs NPU count (cross-scale) ---

agg = df.groupby('npu_count').agg(
    g2_mape=('G2 Error (%)', lambda x: x.abs().mean()),
    au_mape=('AU Error (%)', lambda x: x.abs().mean()),
    g2_mean_err=('G2 Error (%)', 'mean'),
    au_mean_err=('AU Error (%)', 'mean'),
    g2_std_err=('G2 Error (%)', 'std'),
    au_std_err=('AU Error (%)', 'std'),
    count=('workload', 'count'),
).reset_index()

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=agg['npu_count'], y=agg['g2_mape'], mode='lines+markers',
    name='G2 MAPE', marker=dict(size=10), line=dict(color='skyblue', width=3)
))
fig.add_trace(go.Scatter(
    x=agg['npu_count'], y=agg['au_mape'], mode='lines+markers',
    name='Analytical MAPE', marker=dict(size=10), line=dict(color='salmon', width=3)
))
fig.update_layout(
    title='Mean Absolute Percentage Error vs NPU Count',
    xaxis_title='NPU Count', yaxis_title='MAPE (%)',
    xaxis_type='log', xaxis=dict(tickvals=[2,4,8,16,32,64,128]),
    template='plotly_white', font=dict(size=16),
    height=500, width=900,
)
fig.show()

display(agg.style.format({
    'g2_mape': '{:.2f}%', 'au_mape': '{:.2f}%',
    'g2_mean_err': '{:+.2f}%', 'au_mean_err': '{:+.2f}%',
    'g2_std_err': '{:.2f}%', 'au_std_err': '{:.2f}%',
}))

,npu_count,g2_mape,au_mape,g2_mean_err,au_mean_err,g2_std_err,au_std_err,count
0,2,0.64%,0.78%,-0.40%,-0.78%,0.97%,0.96%,38
1,4,3.84%,8.01%,-3.20%,-8.01%,6.35%,11.91%,51
2,8,3.21%,5.13%,-2.50%,-5.12%,4.56%,5.27%,44
3,16,3.45%,38.04%,-1.52%,-38.04%,5.84%,30.83%,35
4,32,6.09%,52.09%,-4.33%,-51.09%,7.79%,35.94%,30
5,64,5.96%,66.77%,-5.85%,-66.31%,6.31%,30.18%,29


In [15]:
# --- Plot 4: Speedup vs NPU count ---

agg_speed = df.groupby('npu_count').agg(
    g2_speedup_mean=('G2 Speedup (x)', 'mean'),
    g2_speedup_min=('G2 Speedup (x)', 'min'),
    g2_speedup_max=('G2 Speedup (x)', 'max'),
    au_speedup_mean=('AU Speedup (x)', 'mean'),
    au_speedup_min=('AU Speedup (x)', 'min'),
    au_speedup_max=('AU Speedup (x)', 'max'),
).reset_index()

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=agg_speed['npu_count'], y=agg_speed['g2_speedup_mean'],
    mode='lines+markers', name='G2 Speedup',
    marker=dict(size=10), line=dict(color='skyblue', width=3),
    error_y=dict(type='data', symmetric=False,
                 array=agg_speed['g2_speedup_max'] - agg_speed['g2_speedup_mean'],
                 arrayminus=agg_speed['g2_speedup_mean'] - agg_speed['g2_speedup_min'])
))
fig.add_trace(go.Scatter(
    x=agg_speed['npu_count'], y=agg_speed['au_speedup_mean'],
    mode='lines+markers', name='Analytical Speedup',
    marker=dict(size=10), line=dict(color='salmon', width=3),
    error_y=dict(type='data', symmetric=False,
                 array=agg_speed['au_speedup_max'] - agg_speed['au_speedup_mean'],
                 arrayminus=agg_speed['au_speedup_mean'] - agg_speed['au_speedup_min'])
))
fig.update_layout(
    title='Simulation Speedup over NS3 vs NPU Count',
    xaxis_title='NPU Count', yaxis_title='Speedup (x)',
    xaxis_type='log', xaxis=dict(tickvals=[2,4,8,16,32,64,128]),
    template='plotly_white', font=dict(size=16),
    height=500, width=900,
)
fig.show()

In [16]:
# --- Plot 5: Error by d_model (model shape sensitivity) ---

if 'd_model' in df.columns:
    for param_name, param_col in [('d_model', 'd_model'), ('num_stacks', 'num_stacks'), ('seq_len', 'seq_len')]:
        if param_col not in df.columns:
            continue
        agg_param = df.groupby(param_col).agg(
            g2_mape=('G2 Error (%)', lambda x: x.abs().mean()),
            au_mape=('AU Error (%)', lambda x: x.abs().mean()),
            count=('workload', 'count'),
        ).reset_index()

        fig = go.Figure()
        fig.add_trace(go.Bar(
            x=agg_param[param_col].astype(str), y=agg_param['g2_mape'],
            name='G2 MAPE', marker_color='skyblue',
            text=agg_param['count'].apply(lambda n: f'n={n}'), textposition='outside'
        ))
        fig.add_trace(go.Bar(
            x=agg_param[param_col].astype(str), y=agg_param['au_mape'],
            name='Analytical MAPE', marker_color='salmon',
        ))
        fig.update_layout(
            title=f'MAPE by {param_name}',
            xaxis_title=param_name, yaxis_title='MAPE (%)',
            barmode='group', template='plotly_white', font=dict(size=16),
            height=450, width=800,
        )
        fig.show()

In [17]:
# --- Plot 6: Spearman rank correlation per NPU count ---

corr_rows = []
for npu in sorted(df['npu_count'].unique()):
    ndf = df[df['npu_count'] == npu]
    if len(ndf) < 3:
        continue
    g2_corr, _ = spearmanr(ndf['NS3_Est_Exec_Time_ns'], ndf['G2_Est_Exec_Time_ns'])
    au_corr, _ = spearmanr(ndf['NS3_Est_Exec_Time_ns'], ndf['Analytical_Est_Exec_Time_ns'])
    corr_rows.append({'npu_count': npu, 'G2 vs NS3': g2_corr, 'Analytical vs NS3': au_corr, 'n': len(ndf)})

if corr_rows:
    corr_df = pd.DataFrame(corr_rows)
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=corr_df['npu_count'], y=corr_df['G2 vs NS3'],
        mode='lines+markers', name='G2 vs NS3',
        marker=dict(size=10), line=dict(color='skyblue', width=3)
    ))
    fig.add_trace(go.Scatter(
        x=corr_df['npu_count'], y=corr_df['Analytical vs NS3'],
        mode='lines+markers', name='Analytical vs NS3',
        marker=dict(size=10), line=dict(color='salmon', width=3)
    ))
    fig.update_layout(
        title="Spearman's Rank Correlation vs NPU Count",
        xaxis_title='NPU Count', yaxis_title='Spearman Correlation',
        xaxis_type='log', xaxis=dict(tickvals=[2,4,8,16,32,64,128]),
        yaxis_range=[0, 1.05],
        template='plotly_white', font=dict(size=16),
        height=500, width=900,
    )
    fig.show()
    display(corr_df)

,npu_count,G2 vs NS3,Analytical vs NS3,n
0,2,0.999562,0.999562,38
1,4,0.999548,0.997828,51
2,8,0.999295,0.999154,44
3,16,0.999160,0.844538,35
4,32,0.997775,0.870078,30
5,64,0.999015,0.890148,29


In [18]:
# --- Summary tables and LaTeX output ---

print(f"\n{'='*80}")
print(f"Overall Summary")
print(f"{'='*80}")
print(f"Total workloads: {len(df)}")
print(f"Overall G2 MAPE: {df['G2 Error (%)'].abs().mean():.2f}%")
print(f"Overall Analytical MAPE: {df['AU Error (%)'].abs().mean():.2f}%")
print(f"Overall G2 Speedup: {df['G2 Speedup (x)'].mean():.2f}x")
print(f"Overall Analytical Speedup: {df['AU Speedup (x)'].mean():.2f}x")

# Per-NPU summary
summary = df.groupby('npu_count').agg(
    count=('workload', 'count'),
    g2_mape=('G2 Error (%)', lambda x: x.abs().mean()),
    au_mape=('AU Error (%)', lambda x: x.abs().mean()),
    g2_speedup=('G2 Speedup (x)', 'mean'),
    au_speedup=('AU Speedup (x)', 'mean'),
    ns3_sim_mean=('NS3_Sim_Time_sec', 'mean'),
    g2_sim_mean=('G2_Sim_Time_sec', 'mean'),
    au_sim_mean=('Analytical_Sim_Time_sec', 'mean'),
).reset_index()

print(f"\n--- Per-NPU Summary ---")
display(summary.style.format({
    'g2_mape': '{:.2f}%', 'au_mape': '{:.2f}%',
    'g2_speedup': '{:.2f}x', 'au_speedup': '{:.2f}x',
    'ns3_sim_mean': '{:.2f}s', 'g2_sim_mean': '{:.2f}s', 'au_sim_mean': '{:.2f}s',
}))

# --- LaTeX: Aggregated by NPU count ---
print(f"\n{'='*80}")
print("LaTeX: Summary by NPU Count")
print(f"{'='*80}")

latex_df = summary.copy()
latex_df.columns = ['NPUs', 'N', 'G2 MAPE (%)', 'AU MAPE (%)',
                     'G2 Speedup', 'AU Speedup',
                     'NS3 Sim (s)', 'G2 Sim (s)', 'AU Sim (s)']

# Add average row
avg_row = pd.DataFrame([{
    'NPUs': '\\textbf{Average}',
    'N': latex_df['N'].sum(),
    'G2 MAPE (%)': df['G2 Error (%)'].abs().mean(),
    'AU MAPE (%)': df['AU Error (%)'].abs().mean(),
    'G2 Speedup': df['G2 Speedup (x)'].mean(),
    'AU Speedup': df['AU Speedup (x)'].mean(),
    'NS3 Sim (s)': df['NS3_Sim_Time_sec'].mean(),
    'G2 Sim (s)': df['G2_Sim_Time_sec'].mean(),
    'AU Sim (s)': df['Analytical_Sim_Time_sec'].mean(),
}])
latex_df = pd.concat([latex_df, avg_row], ignore_index=True)

latex_str = latex_df.to_latex(
    index=False,
    formatters={
        'G2 MAPE (%)': '{:.2f}\\%'.format,
        'AU MAPE (%)': '{:.2f}\\%'.format,
        'G2 Speedup': '{:.2f}x'.format,
        'AU Speedup': '{:.2f}x'.format,
        'NS3 Sim (s)': '{:.2f}'.format,
        'G2 Sim (s)': '{:.2f}'.format,
        'AU Sim (s)': '{:.2f}'.format,
    },
    caption='Simulator Accuracy and Speed Summary by NPU Count (Experiment 8).',
    label='tab:exp8_summary',
    position='!htbp',
    column_format='rrccccccc',
    escape=False,
)
lines = latex_str.splitlines()
lines.insert(-2, '\\hline')
latex_str = '\n'.join(lines)
latex_str = latex_str.replace('\\toprule', '\\hline').replace('\\midrule', '\\hline').replace('\\bottomrule', '\\hline')
print(latex_str)


Overall Summary
Total workloads: 227
Overall G2 MAPE: 3.69%
Overall Analytical MAPE: 24.20%
Overall G2 Speedup: 111.26x
Overall Analytical Speedup: 258.65x

--- Per-NPU Summary ---


,npu_count,count,g2_mape,au_mape,g2_speedup,au_speedup,ns3_sim_mean,g2_sim_mean,au_sim_mean
0,2,38,0.64%,0.78%,30.14x,48.89x,60.30s,1.64s,1.11s
1,4,51,3.84%,8.01%,118.40x,198.87x,434.25s,3.87s,2.55s
2,8,44,3.21%,5.13%,144.97x,247.75x,1427.06s,10.23s,6.69s
3,16,35,3.45%,38.04%,172.90x,330.73x,1333.70s,12.18s,7.15s
4,32,30,6.09%,52.09%,124.81x,431.88x,2246.27s,21.79s,4.62s
5,64,29,5.96%,66.77%,65.40x,389.01x,3884.30s,171.46s,13.03s



LaTeX: Summary by NPU Count
\begin{table}[!htbp]
\caption{Simulator Accuracy and Speed Summary by NPU Count (Experiment 8).}
\label{tab:exp8_summary}
\begin{tabular}{rrccccccc}
\hline
NPUs & N & G2 MAPE (%) & AU MAPE (%) & G2 Speedup & AU Speedup & NS3 Sim (s) & G2 Sim (s) & AU Sim (s) \\
\hline
2 & 38 & 0.64\% & 0.78\% & 30.14x & 48.89x & 60.30 & 1.64 & 1.11 \\
4 & 51 & 3.84\% & 8.01\% & 118.40x & 198.87x & 434.25 & 3.87 & 2.55 \\
8 & 44 & 3.21\% & 5.13\% & 144.97x & 247.75x & 1427.06 & 10.23 & 6.69 \\
16 & 35 & 3.45\% & 38.04\% & 172.90x & 330.73x & 1333.70 & 12.18 & 7.15 \\
32 & 30 & 6.09\% & 52.09\% & 124.81x & 431.88x & 2246.27 & 21.79 & 4.62 \\
64 & 29 & 5.96\% & 66.77\% & 65.40x & 389.01x & 3884.30 & 171.46 & 13.03 \\
\textbf{Average} & 227 & 3.69\% & 24.20\% & 111.26x & 258.65x & 1383.00 & 29.79 & 5.43 \\
\hline
\hline
\end{tabular}
\end{table}
